<a href="https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/expdpy_explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Exploring Cambodia's subnational panels with `geo4cambodia` + `expdpy`

Part of the [GeoDevelopment Observatory of Cambodia](https://quarcs-lab.github.io/geo4cambodia/).

Two packages, one workflow:

- **[geo4cambodia](https://github.com/quarcs-lab/geo4cambodia)** loads the observatory's panels (one row per unit and year) with unit names, the provinces or regions they belong to, and labelled variables.
- **[expdpy](https://cmg777.github.io/expdpy/)** explores panel data: structure, descriptive tables, within/between variation, trajectories, distributions, transitions, missing values and correlations. Every result has a figure or table and a plain-language reading (`.interpret()`).

We explore three panels at three spatial scales:

| Panel | Scale | Years | Section |
| --- | --- | --- | --- |
| `gdppc`: GDP per capita | 202 districts | 1990–2022 | 1 |
| `env`: population, lights, electricity-based GDP, PM2.5 | 25 provinces | 1990–2023 | 2 |
| `hdi`: human development, income, lights | 17 Global Data Lab regions | 2013–2019 | 3 |

The companion notebook [`expdpy_analyze.ipynb`](https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/expdpy_analyze.ipynb) runs panel econometrics on the same data.

## 0. Setup

In [1]:
import importlib.util

missing = [p for p in ("geo4cambodia", "expdpy") if importlib.util.find_spec(p) is None]
if missing:
    !pip install -q {" ".join(missing)}

In [2]:
import warnings

import numpy as np
import geo4cambodia as geo
import expdpy as ex

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*showing .* of")   # expdpy notes when it samples units for readability

`geo.for_expdpy()` returns the two inputs expdpy expects: the panel `df` and its data dictionary `df_def` (labels, units and the role of each column), built from the observatory's metadata. `groups=["province"]` adds the name of the province each district lies in. `ex.set_labels(..., set_panel=True)` then declares `geo_id` × `year` as the panel, so later calls need no `entity=` or `time=`.

In [3]:
df, df_def = geo.for_expdpy("gdppc", groups=["province"])
df = ex.set_labels(df, df_def, set_panel=True)
df_def

,var_name,var_def,label,type,role,can_be_na
0,geo_id,geo ID,geo ID,entity,,False
1,name,Name,Unit name,factor,entity_name,True
2,year,Year,Year,time,,False
3,province,Name of the Province the unit lies in,Province name,factor,,True
4,GID_2,GADM level-2 code,GADM level-2 code,factor,,True
5,y,"GDP per capita (PPP, constant 2017 internation...",GDP per capita,numeric,,True
6,ln_y,Log GDP per capita (log),Log GDP per capita,numeric,,True
7,trend6_ln_y,Hodrick–Prescott trend of log GDP per capita (...,Hodrick–Prescott trend of log GDP per capita (...,numeric,,True
8,trend100_ln_y,Hodrick–Prescott trend of log GDP per capita (...,Hodrick–Prescott trend of log GDP per capita (...,numeric,,True
9,trend400_ln_y,Hodrick–Prescott trend of log GDP per capita (...,Hodrick–Prescott trend of log GDP per capita (...,numeric,,True


## 1. Districts: GDP per capita, 1990–2022

### Structure and summary

In [4]:
structure = ex.explore_panel_structure(df)
print(structure.interpret())
structure.fig

The panel has **202 units** observed over **33 periods**.

It is **balanced**: every unit appears in every period, with no gaps.

Unbalanced panels and interior gaps reduce the information a within (fixed-effects) estimator can use, and shifting unit composition over time can drive apparent trends.


In [5]:
ex.explore_descriptive_table(df[["y", "ln_y"]]).gt

GT(_tbl_data=             Variable         Mean    Std. dev.      Median        Min.  \
0      GDP per capita  2109.472247  1726.687692  1623.50000  436.000000   
1  Log GDP per capita     7.428151     0.637674     7.39234    6.077642   

           Max.  
0  12608.000000  
1      9.442087  , _body=<great_tables._gt_data.Body object at 0x122572010>, _boxhead=Boxhead([ColInfo(var='Variable', type=<ColInfoTypeEnum.stub: 2>, column_label='Variable', column_align='left', column_width=None, stub_level=0), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='right', column_width=None, stub_level=None), ColInfo(var='Std. dev.', type=<ColInfoTypeEnum.default: 1>, column_label='Std. dev.', column_align='right', column_width=None, stub_level=None), ColInfo(var='Median', type=<ColInfoTypeEnum.default: 1>, column_label='Median', column_align='right', column_width=None, stub_level=None), ColInfo(var='Min.', type=<ColInfoTypeEnum.default: 1>, column_label='Min.', column_align='right', column_width=None, stub_level=None), ColInfo(var='Max.', type=<ColInfoTypeEnum.default: 1>, column_label='Max.', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12336f750>, _spanners=Spanners([]), _heading=Heading(title='Descriptive Statistics', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x121f80990>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x123378b10>, _source_notes=['Observations: 6,666.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x123378b50>, _formats=[<great_tables._gt_data.FormatInfo object at 0x123328750>, <great_tables._gt_data.FormatInfo object at 0x12336f490>, <great_tables._gt_data.FormatInfo object at 0x12338da10>, <great_tables._gt_data.FormatInfo object at 0x12338e6d0>, <great_tables._gt_data.FormatInfo object at 0x12338e3d0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss

### Within and between variation

Do districts differ mostly from *each other* (between), or does each district change mostly *over time* (within)? The answer tells you how much fixed-effects models can learn from the data.

In [6]:
xt = ex.explore_xtsum_table(df, var=["y", "ln_y"])
print(xt.interpret())
xt.gt

Splitting each variable's variation into **between** (differences across units) and **within** (variation over time inside a unit):
- **y**: between SD 1.31e+03, within SD 1.13e+03 — variation is split between cross-unit and over-time.
- **ln_y**: between SD 0.416, within SD 0.484 — variation is split between cross-unit and over-time.

_Between variation drives cross-sectional comparisons; within variation is what fixed-effects (within) estimators rely on._


GT(_tbl_data=             variable Statistic         Mean    Std. dev.         Min.  \
0      GDP per capita   Overall  2109.472247  1726.687692   436.000000   
1      GDP per capita   Between          NaN  1306.717206  1211.545455   
2      GDP per capita    Within          NaN  1132.315603 -2035.679268   
3  Log GDP per capita   Overall     7.428151     0.637674     6.077642   
4  Log GDP per capita   Between          NaN     0.415593     6.958293   
5  Log GDP per capita    Within          NaN     0.484499     6.221904   

           Max.       N  n units  T-bar  
0  12608.000000  6666.0    202.0   33.0  
1   7210.757576     NaN      NaN    NaN  
2   7755.229823     NaN      NaN    NaN  
3      9.442087  6666.0    202.0   33.0  
4      8.800703     NaN      NaN    NaN  
5      8.732031     NaN      NaN    NaN  , _body=<great_tables._gt_data.Body object at 0x1233925d0>, _boxhead=Boxhead([ColInfo(var='variable', type=<ColInfoTypeEnum.row_group: 3>, column_label='variable', column_align='left', column_width=None, stub_level=None), ColInfo(var='Statistic', type=<ColInfoTypeEnum.stub: 2>, column_label='Statistic', column_align='left', column_width=None, stub_level=0), ColInfo(var='Mean', type=<ColInfoTypeEnum.default: 1>, column_label='Mean', column_align='right', column_width=None, stub_level=None), ColInfo(var='Std. dev.', type=<ColInfoTypeEnum.default: 1>, column_label='Std. dev.', column_align='right', column_width=None, stub_level=None), ColInfo(var='Min.', type=<ColInfoTypeEnum.default: 1>, column_label='Min.', column_align='right', column_width=None, stub_level=None), ColInfo(var='Max.', type=<ColInfoTypeEnum.default: 1>, column_label='Max.', column_align='right', column_width=None, stub_level=None), ColInfo(var='N', type=<ColInfoTypeEnum.default: 1>, column_label='N', column_align='right', column_width=None, stub_level=None), ColInfo(var='n units', type=<ColInfoTypeEnum.default: 1>, column_label='n units', column_align='right', column_width=None, stub_level=None), ColInfo(var='T-bar', type=<ColInfoTypeEnum.default: 1>, column_label='T-bar', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12332b990>, _spanners=Spanners([]), _heading=Heading(title='Within/Between Variation (xtsum)', subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x123392690>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1233926d0>, _source_notes=['Overall = pooled. Between = across unit means. Within = over time inside a unit (deviations from the unit mean, recentered on the grand mean).'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x123392710>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1233929d0>, <great_tables._gt_data.FormatInfo object at 0x123392c50>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x123392e90>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_

### Trajectories

One line per district, and the quantiles of the distribution over time:

In [7]:
ex.explore_spaghetti_plot(df, "ln_y").fig

In [8]:
ex.explore_quantile_trend_plot(df, var="y").fig

Average paths of four contrasting provinces: the capital, the tourism hub around Angkor, the north-western rice bowl, and the remote north-east.

In [9]:
four = df[df["province"].isin(["Phnom Penh", "Siem Reap", "Battambang", "Ratanak Kiri"])]
ex.explore_trend_plot_by_group(four, group_var="province", var="ln_y").fig

### Distribution dynamics

How the whole distribution moved, and how often districts change quintile from one year to the next:

In [10]:
dist = ex.explore_distribution_over_time(df, "ln_y")
print(dist.interpret())
dist.fig

Tracking the distribution of **ln_y** across periods:
- The median moved higher (from 6.64 to 8.05).
- The spread (IQR) widened (from 0.322 to 0.347).


In [11]:
trans = ex.explore_transition_matrix(df, "y")
print(trans.interpret())
trans.fig

Across 5 states, the average **persistence** (probability of staying in the same state from one period to the next) is 0.798.
- Stickiest: **Q5** (stays with probability 0.944).
- Least sticky: **Q2** (stays with probability 0.713).


## 2. Provinces: lights, population, electricity-based GDP and PM2.5

The `env` variables are **zonal sums** over 1 km grid cells. We derive comparable per-person and average indicators in pandas:

- `lights_pc`: VIIRS-like lights per 1,000 people
- `egdp_pc`: electricity-based GDP per person
- `pm25`: approximate mean PM2.5 concentration, the zonal sum divided by the number of 0.01° (≈ 1.2 km²) cells in the province

The unit table supplies each province's area; `groups=["gdl"]` adds its Global Data Lab region.

In [12]:
env, env_def = geo.for_expdpy("env", groups=["gdl"])
provinces = geo.boundaries("province").merge(geo.units("province")[["geo_id", "area_km2"]], on="geo_id")
lat = provinces.to_crs("EPSG:32648").centroid.to_crs("EPSG:4326").y
cell_km2 = (111.32 * 0.01) ** 2 * np.cos(np.radians(lat))
cells = dict(zip(provinces["geo_id"], provinces["area_km2"].astype(float) / cell_km2))

env["lights_pc"] = env["total_VNL"] / env["total_GlobPOP_Count"] * 1000
env["egdp_pc"] = env["total_EGDP"] / env["total_GlobPOP_Count"]
env["pm25"] = env["total_GHAP"] / env["geo_id"].map(cells)

labels = {"lights_pc": "Night-time lights per 1,000 people", "egdp_pc": "Electricity-based GDP per person",
          "pm25": "Mean PM2.5 (µg/m³, approx.)"}
env, env_def = geo.for_expdpy(env, labels=labels)   # your own panel: published variables keep their labels
env = ex.set_labels(env, env_def, set_panel=True)

Each source covers different years, so the panel has many gaps. They are not errors:

In [13]:
miss = ex.explore_missing_values_plot(env.drop(columns=["shapeISO", "total_NPP"]))
print(miss.interpret())
miss.fig

Across 11 variables and 38 levels, overall **29.4%** of cells are missing (so the data are 70.6% complete).
The variable with the most missingness is **total_GHAP** (84.2% missing); the level with the most missingness is **1986** (72.7% missing).


In [14]:
cols = ["lights_pc", "egdp_pc", "pm25", "total_GlobPOP_Count"]
corr = ex.explore_correlation_table(env[cols])
print(corr.interpret())
corr.gt

The strongest linear association is between **pm25** and **total_GlobPOP_Count** (Pearson r = 0.638, statistically significant at the 1% level).

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


GT(_tbl_data=                                       term         A          B          C  \
0     A: Night-time lights per 1,000 people                -0.00      -0.08   
1       B: Electricity-based GDP per person  **0.26**             **-0.32**   
2            C: Mean PM2.5 (µg/m³, approx.)     -0.08      -0.22              
3  D: Zonal sum of gridded population count  **0.18**  **-0.52**   **0.63**   

           D  
0   **0.23**  
1  **-0.49**  
2   **0.64**  
3             , _body=<great_tables._gt_data.Body object at 0x123598950>, _boxhead=Boxhead([ColInfo(var='term', type=<ColInfoTypeEnum.stub: 2>, column_label='term', column_align='left', column_width=None, stub_level=0), ColInfo(var='A', type=<ColInfoTypeEnum.default: 1>, column_label='A', column_align='right', column_width=None, stub_level=None), ColInfo(var='B', type=<ColInfoTypeEnum.default: 1>, column_label='B', column_align='right', column_width=None, stub_level=None), ColInfo(var='C', type=<ColInfoTypeEnum.default: 1>, column_label='C', column_align='right', column_width=None, stub_level=None), ColInfo(var='D', type=<ColInfoTypeEnum.default: 1>, column_label='D', column_align='right', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12364b090>, _spanners=Spanners([]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x123610990>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x1235e4410>, _source_notes=['This table reports Pearson correlations above and Spearman correlations below the diagonal. The number of observations ranges from 75 to 825. Correlations with significance levels below 5% appear in bold.'], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x1235e4b90>, _formats=[<great_tables._gt_data.FormatInfo object at 0x1235e44d0>], _substitutions=[], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_right_color=OptionsInfo(scss=True, category='table', type='value', value='#D3D3D3'), t

**Within vs between.** Is the association between lights and electricity-based GDP per person the same across provinces (between) as within a province over time (within)? Decomposing it shows why pooled regressions can mislead:

In [15]:
wb = ex.explore_scatter_plot_within_between(env.dropna(subset=["lights_pc", "egdp_pc"]), x="lights_pc", y="egdp_pc")
print(wb.interpret())
wb.fig

The pooled association between **Night-time lights per 1,000 people** and **Electricity-based GDP per person** has slope -1.42e-05. It blends a **between-unit** slope of -0.000349 (comparing unit averages) with a **within-unit** slope of 0.000327 (variation over time inside a unit).
The two diverge markedly — the cross-unit and over-time relationships tell different stories, a sign that unit-level confounders matter and that a fixed-effects specification would change the estimate.

_These are associations, not causal effects. A causal reading needs a research design — see `explain('correlation_vs_causation')`._


## 3. Global Data Lab regions: human development, 2013–2019

In [16]:
hdi, hdi_def = geo.for_expdpy("hdi")
hdi = ex.set_labels(hdi, hdi_def, set_panel=True)

trend = ex.explore_trend_plot(hdi, var=["HDI"])
print(trend.interpret())
trend.fig

Over the observed period:
- **HDI** moved higher, from 0.553 in 2013 to 0.59 in 2019 (change of 0.0372).


In [17]:
ex.explore_bar_plot_by_group(hdi[hdi["year"] == 2019], by_var="name", var="HDI", order_by_stat=True).fig

Which region-years stand out?

In [18]:
ex.explore_ext_obs_table(hdi, var="HDI").gt

GT(_tbl_data=                                           geo_id  year    HDI           
0                          Phnom Penh (KH-GDL-08)  2019  0.731  Highest 5
1                          Phnom Penh (KH-GDL-08)  2018  0.718  Highest 5
2                          Phnom Penh (KH-GDL-08)  2017  0.714  Highest 5
3                          Phnom Penh (KH-GDL-08)  2016  0.708  Highest 5
4                          Phnom Penh (KH-GDL-08)  2013  0.702  Highest 5
5  Preah Vihear-Stueng Traeng-Kracheh (KH-GDL-15)  2015  0.499   Lowest 5
6            Mondol Kiri-Rotanak Kiri (KH-GDL-16)  2014  0.497   Lowest 5
7  Preah Vihear-Stueng Traeng-Kracheh (KH-GDL-15)  2014  0.495   Lowest 5
8  Preah Vihear-Stueng Traeng-Kracheh (KH-GDL-15)  2013  0.492   Lowest 5
9            Mondol Kiri-Rotanak Kiri (KH-GDL-16)  2013  0.487   Lowest 5, _body=<great_tables._gt_data.Body object at 0x12360c1d0>, _boxhead=Boxhead([ColInfo(var='geo_id', type=<ColInfoTypeEnum.default: 1>, column_label='geo ID', column_align='left', column_width=None, stub_level=None), ColInfo(var='year', type=<ColInfoTypeEnum.default: 1>, column_label='Year', column_align='right', column_width=None, stub_level=None), ColInfo(var='HDI', type=<ColInfoTypeEnum.default: 1>, column_label='Subnational Human Development Index', column_align='right', column_width=None, stub_level=None), ColInfo(var='  ', type=<ColInfoTypeEnum.row_group: 3>, column_label='  ', column_align='left', column_width=None, stub_level=None)]), _stub=<great_tables._gt_data.Stub object at 0x12360ded0>, _spanners=Spanners([]), _heading=Heading(title=None, subtitle=None, preheader=None), _stubhead=None, _summary_rows=<great_tables._gt_data.SummaryRows object at 0x12360c890>, _summary_rows_grand=<great_tables._gt_data.SummaryRows object at 0x12360fdd0>, _source_notes=[], _footnotes=[], _styles=[], _locale=<great_tables._gt_data.Locale object at 0x12360f810>, _formats=[<great_tables._gt_data.FormatInfo object at 0x12360d3d0>], _substitutions=[<great_tables._gt_data.FormatInfo object at 0x12360ed10>], _col_merge=[], _transforms=[], _options=Options(table_id=OptionsInfo(scss=False, category='table', type='value', value=None), table_caption=OptionsInfo(scss=False, category='table', type='value', value=None), table_width=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_layout=OptionsInfo(scss=True, category='table', type='value', value='fixed'), table_margin_left=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_margin_right=OptionsInfo(scss=True, category='table', type='px', value='auto'), table_background_color=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_additional_css=OptionsInfo(scss=False, category='table', type='values', value=[]), table_font_names=OptionsInfo(scss=False, category='table', type='values', value=['-apple-system', 'BlinkMacSystemFont', 'Segoe UI', 'Roboto', 'Oxygen', 'Ubuntu', 'Cantarell', 'Helvetica Neue', 'Fira Sans', 'Droid Sans', 'Arial', 'sans-serif']), table_font_size=OptionsInfo(scss=True, category='table', type='px', value='16px'), table_font_weight=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_style=OptionsInfo(scss=True, category='table', type='value', value='normal'), table_font_color=OptionsInfo(scss=True, category='table', type='value', value='#333333'), table_font_color_light=OptionsInfo(scss=True, category='table', type='value', value='#FFFFFF'), table_border_top_include=OptionsInfo(scss=False, category='table', type='boolean', value=True), table_border_top_style=OptionsInfo(scss=True, category='table', type='value', value='solid'), table_border_top_width=OptionsInfo(scss=True, category='table', type='px', value='2px'), table_border_top_color=OptionsInfo(scss=True, category='table', type='value', value='#A8A8A8'), table_border_right_style=OptionsInfo(scss=True, category='table', type='value', value='none'), table_border_right_width=OptionsInfo(scss=True, category='table', type=

## Next steps

- **Panel econometrics** on these data (fixed vs random effects, Hausman tests, clustered standard errors, convergence): [`expdpy_analyze.ipynb`](https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/expdpy_analyze.ipynb)
- **Spatial analysis** with geometrics: [`geometrics_convergence.ipynb`](https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/geometrics_convergence.ipynb)
- Explore your own indicator: build it with `geo.link()`, `geo.zonal()` or `geo.combine()` ([quickstart](https://colab.research.google.com/github/quarcs-lab/geo4cambodia/blob/main/notebooks/geo4cambodia_quickstart.ipynb)), then `geo.for_expdpy(my_panel, labels={...})`.
- No code at all: expdpy's [Explore app](https://expdpy-explore.streamlit.app) accepts any CSV, including the observatory's files.